In [34]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import pickle
import math

from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path('..').resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'

sys.path.append(str(PROJECT_ROOT))
import utils_recommender as ur

In [35]:
ratings_full = pd.read_csv(DATA_PROCESSED / 'ratings_full_with_predictions.csv')
courses = pd.read_csv(DATA_PROCESSED / 'cleaned_courses.csv')

train_df, test_df = ur.train_test_split_by_user(
    ratings_full,
    min_user_ratings=5,
    test_size=0.2,
    seed=42
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (189876, 3)
Test : (43611, 3)


In [36]:
# Content-Based
with open(MODELS_DIR / 'tfidf_similarity.pkl', 'rb') as f:
    tfidf_sim = pickle.load(f)

# Collaborative Filtering (PRECOMPUTED)
with open(MODELS_DIR / '06_pred_user_knn.pkl', 'rb') as f:
    pred_user_knn = pickle.load(f)

print("✅ Models loaded")

✅ Models loaded


In [37]:
course_ids = courses['COURSE_ID'].tolist()
course_index = {cid: idx for idx, cid in enumerate(course_ids)}

In [38]:
def predict_content(user_id, item_id, k=5):
    if item_id not in tfidf_sim.columns:
        return np.nan

    sim_scores = tfidf_sim[item_id]

    user_history = train_df[train_df['user'] == user_id]

    sims = []
    for _, row in user_history.iterrows():
        if row['item'] in sim_scores.index:
            sims.append((sim_scores[row['item']], row['rating']))

    if len(sims) == 0:
        return np.nan

    sims = sorted(sims, reverse=True)[:k]

    num = sum(sim * rating for sim, rating in sims)
    den = sum(abs(sim) for sim, _ in sims)

    return num / den if den != 0 else np.nan


In [39]:
def predict_cf(user_id, item_id):
    try:
        if user_id in pred_user_knn and item_id in pred_user_knn[user_id].index:
            return pred_user_knn[user_id][item_id]
        return np.nan
    except:
        return np.nan


In [40]:
ALPHA = 0.5  # CF stronger

def hybrid_predict(user_id, item_id):
    cf_pred = predict_cf(user_id, item_id)
    cb_pred = predict_content(user_id, item_id)

    if np.isnan(cf_pred) and np.isnan(cb_pred):
        return np.nan
    elif np.isnan(cf_pred):
        return cb_pred
    elif np.isnan(cb_pred):
        return cf_pred
    else:
        return ALPHA * cf_pred + (1 - ALPHA) * cb_pred

In [41]:
preds, truths = [], []

for _, row in test_df.iterrows():
    pred = hybrid_predict(row['user'], row['item'])
    if not np.isnan(pred):
        preds.append(pred)
        truths.append(row['rating'])

print("Valid predictions:", len(preds))

Valid predictions: 43611


In [42]:
mae = mean_absolute_error(truths, preds)
rmse = math.sqrt(mean_squared_error(truths, preds))

print("MAE :", mae)
print("RMSE:", rmse)

MAE : 0.444253813250947
RMSE: 0.8946643461797994


In [43]:
def get_top_n(user_id, n=10):
    items = courses['COURSE_ID'].tolist()
    scores = []

    for item in items:
        score = hybrid_predict(user_id, item)
        if not np.isnan(score):
            scores.append((item, score))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return [i[0] for i in scores[:n]]

In [44]:
def evaluate_ranking(test_df, n=10):
    recalls, hits, ndcgs = [], [], []

    users = test_df['user'].unique()[:200]

    for user in users:
        true_items = test_df[test_df['user'] == user]['item'].tolist()
        if len(true_items) == 0:
            continue

        recs = get_top_n(user, n)
        hit_set = set(recs) & set(true_items)

        # Recall
        recalls.append(len(hit_set) / len(true_items))

        # Hit Rate
        hits.append(1 if len(hit_set) > 0 else 0)

        # NDCG
        dcg = sum(1/np.log2(i+2) for i, item in enumerate(recs) if item in true_items)
        idcg = sum(1/np.log2(i+2) for i in range(min(len(true_items), n)))
        ndcgs.append(dcg/idcg if idcg > 0 else 0)

    return np.mean(recalls), np.mean(hits), np.mean(ndcgs)

In [45]:
recall, hitrate, ndcg = evaluate_ranking(test_df)

print("Recall@10 :", recall)
print("HitRate@10:", hitrate)
print("NDCG@10   :", ndcg)

Recall@10 : 0.3451388888888889
HitRate@10: 0.455
NDCG@10   : 0.1822879398864513


In [46]:
results = pd.DataFrame([{
    "Model": "Hybrid (TFIDF + UserKNN)",
    "MAE": mae,
    "RMSE": rmse,
    "Recall@10": recall,
    "HitRate@10": hitrate,
    "NDCG@10": ndcg
}])

results

,Model,MAE,RMSE,Recall@10,HitRate@10,NDCG@10
0,Hybrid (TFIDF + UserKNN),0.444254,0.894664,0.345139,0.455,0.182288


In [47]:
results.to_csv(RESULTS_DIR / "hybrid_results.csv", index=False)
print("✅ Saved hybrid_results.csv")

✅ Saved hybrid_results.csv
